In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:gemma4",
    context_schema=ColourContext  
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    
    # We can provide context schema to the agent, which the agent can then draw on to inform its actions
    # See here for docs: https://docs.langchain.com/oss/python/concepts/context
    # However, to access the context, we need to make tools calls (via ToolRuntime), to ensure only the relevant context is retrieved - this is done to not overload the agent with context
    # See Accessing Context section
    context=ColourContext()
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='dc9365cc-7b4a-4d3b-87c4-fc663de4e10a'),
              AIMessage(content="I don't know! As an AI, I don't have access to your personal thoughts or preferences, so I can't tell what your favorite color is.\n\nYou'll have to tell me! 😊", additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T06:09:12.619087Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17621069667, 'load_duration': 8120094333, 'prompt_eval_count': 22, 'prompt_eval_duration': 207413000, 'eval_count': 263, 'eval_duration': 9289759000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13907-30d3-78e0-b9a0-9e765ff4e9b9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 263, 'total_tokens': 285})]}


## Accessing Context

In [19]:
from langchain.tools import tool, ToolRuntime

# ToolRuntime docs: https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    # from the docs above, "context: Runtime context (shared with Runtime)", so we're accessing the runtime, going into context which was provided to the tool, and retrieving favourite_colour from the ColourContext object
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [20]:
agent = create_agent(
    model="ollama:gemma4",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [21]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='14676550-8370-4605-a0da-837f985da03e'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T06:24:53.332119Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6011276583, 'load_duration': 3742666, 'prompt_eval_count': 87, 'prompt_eval_duration': 140547000, 'eval_count': 166, 'eval_duration': 5864814000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13915-b8d6-79e1-a939-5e0bb6af72b4-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'b56d7ae8-0bfc-4c8a-8fb8-c711cc96488a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 166, 'total_tokens': 253}),
              ToolMessage(content='red', name='get_favourite_colour', id='a8efc194-0e68-4ce7-9e5a-392974c061ea', tool_call_id='b56d7ae8-0bfc-4c8a-8fb

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='c9650218-0117-4820-9d34-8c5c95fcf298'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T06:21:16.171265Z', 'done': True, 'done_reason': 'stop', 'total_duration': 536234250, 'load_duration': 925833, 'prompt_eval_count': 87, 'prompt_eval_duration': 131408999, 'eval_count': 12, 'eval_duration': 401118000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13912-7df2-7c02-b351-015c9f42648d-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': '55201325-0aa0-4e33-9e36-d6f1a1397081', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 12, 'total_tokens': 99}),
              ToolMessage(content='green', name='get_favourite_colour', id='89f6ccbb-f312-452d-866e-a34b00c7ba7d', tool_call_id='55201325-0aa0-4e33-9e36-d6